# Lab 1 - Guvenilir Cikti

**Kanitladigi tez:** Model ciktisini ayristirilabilir ve dogrulanabilir
kilmadan sisteme sokamazsiniz.

Bu labda su sirayla ilerleyecegiz:

1. Ayni istegi bes kez calistirip ciktinin degistigini gorecegiz
2. Sicakligi sifira indirip ne degistigini olcecegiz
3. Ciktiyi bir semaya zorlayacagiz
4. Sema bozuldugunda sistemin bunu nasil yakaladigini gorecegiz
5. Onarim turu ekleyip bozuk ciktiyi tek turda duzeltecegiz

## Kurulum

Asagidaki iki hucreyi sirayla calistirin. Bilgisayariniza hicbir sey
kurulmuyor: her sey sizin Colab calisma zamaninizda calisir ve
oturum kapaninca silinir.

In [ ]:
# 1/2 - Repoyu indir
!git clone -q https://github.com/silexi/guvenli-ai-mimarileri-lab.git 2>/dev/null || echo 'repo zaten var'
%cd -q guvenli-ai-mimarileri-lab
!pip install -q -U transformers accelerate 2>/dev/null
print('kurulum tamam')

In [ ]:
# 2/2 - Modeli sec ve yukle
import os, sys
sys.path.insert(0, '.')

# Uc secenek:
#   'colab' -> kendi calisma zamaninizda kucuk bir model (varsayilan)
#   'mock'  -> model yuklemeden, kayitli cevaplarla (yedek yol)
os.environ['LAB_SAGLAYICI'] = 'colab'

from ortak import llm
print(llm.durum())

# Modeli simdi yukleyelim ki sonraki hucreler beklemesin.
# GPU yoksa bu adim birkac dakika surebilir.
try:
    llm.model_yukle()
except Exception as hata:
    print('Model yuklenemedi:', hata)
    print('MOCK moda geciliyor, lab yapisi aynen calisacak.')
    os.environ['LAB_SAGLAYICI'] = 'mock'

print('\nLab 1 icin hazir.')

---
## 1. Ayni istek, bes calistirma

Su destek talebini modele bes kez vereceğiz. Prompt her seferinde
**birebir ayni**. Tahmininiz: kac tanesi birbirinin aynisi olacak?

In [ ]:
TALEP = '''Faturam iki kez kesildi, yarina kadar cozulmezse
iptal etmek istiyorum. Agustos donemi, yaklasik 12.400 TL fazla.'''

ISTEK = f'''Asagidaki destek talebini kategoriye ata ve tek cumlede
gerekce yaz.

Talep: {TALEP}'''

print(ISTEK)

In [ ]:
from ortak import llm

cevaplar = []
for tur in range(1, 6):
    c = llm.sor(ISTEK, sicaklik=0.7, senaryo='lab1_serbest')
    cevaplar.append(c.metin)
    print(f'--- Tur {tur} ({c.sure_sn:.1f} sn) ---')
    print(c.metin)
    print()

In [ ]:
# Kac tanesi birbirinin aynisi?
benzersiz = set(cevaplar)
print(f'5 turda {len(benzersiz)} farkli cevap uretildi.')
print()
print('Bu cevaplarin hepsini ayni kodla ayristirabilir miydiniz?')

> **Durup dusunun.** Bu cevaplarin hepsi insan icin okunabilir.
> Ama entegrasyonunuz bu metinden tutari, donemi ve aciligi cikarmak
> zorunda. Duzenli ifade (regex) ile bunu yapmayi denerseniz, her yeni
> ifade bicimi kodunuzu kirar.

---
## 2. Sicaklik 0: kararlilik artar, garanti gelmez

In [ ]:
cevaplar_sifir = []
for tur in range(1, 6):
    c = llm.sor(ISTEK, sicaklik=0.0, senaryo='lab1_serbest')
    cevaplar_sifir.append(c.metin)

print(f'Sicaklik 0.7 -> {len(set(cevaplar))} farkli cevap')
print(f'Sicaklik 0.0 -> {len(set(cevaplar_sifir))} farkli cevap')
print()
print('Sicaklik 0 DAHA KARARLI demektir, DETERMINISTIK demek degil.')

---
## 3. Ciktiyi semaya zorlayalim

Simdi ayni isi yapiyoruz ama modelden serbest metin degil,
belirli alanlari olan bir JSON istiyoruz.

In [ ]:
from ortak import sema

print('Beklenen sema:')
print(sema.sema_metni())

In [ ]:
ISTEK_SEMALI = f'''Asagidaki destek talebinden bilgileri cikar.

Talep: {TALEP}

Yalnizca su semaya uyan bir JSON nesnesi dondur. Aciklama yazma:
{sema.sema_metni()}'''

c = llm.sor(ISTEK_SEMALI, sicaklik=0.3, senaryo='lab1_sema')
print(c.metin)

---
## 4. Dogrulama: sistem bozuk ciktiyi yakaliyor mu?

Bes tur calistirip her turda semayi dogrulayacagiz. Bazi turlarin
**basarisiz olmasini bekliyoruz**. Onemli olan basarisizligin
sessizce gecmemesi, yakalanmasi.

In [ ]:
gecen, kalan = 0, 0

for tur in range(1, 6):
    c = llm.sor(ISTEK_SEMALI, sicaklik=0.7, senaryo='lab1_sema')
    veri = llm.json_ayikla(c.metin)
    try:
        temiz = sema.dogrula(veri)
        gecen += 1
        print(f'Tur {tur}: GECTI')
        print(f'   {temiz}')
    except sema.SemaHatasi as hata:
        kalan += 1
        print(f'Tur {tur}: YAKALANDI')
        print(f'   Ham cikti : {c.metin[:80]}')
        print(f'   Hata      : {hata}')
    print()

print(f'Gecen: {gecen}  |  Yakalanan: {kalan}')

> **Kritik nokta.** Yakalanan turlar bir arıza degil, tasarimin
> calistiginin kanitidir. Sema dogrulamasi olmasaydi bu bozuk kayitlar
> sisteme girer ve hata cok daha sonra, cok daha pahali bir yerde
> ortaya cikardi.

---
## 5. Onarim turu

Sema bozuldugunda bastan uretmek yerine, hatayi modele geri verip
tek turda duzelttirmek hem ucuz hem daha basarilidir.

In [ ]:
import json

def semali_cikar(istek, azami_deneme=2):
    """Sema dogrulamasi ve onarim turu olan cikarim."""
    c = llm.sor(istek, sicaklik=0.3, senaryo='lab1_sema')
    ham = c.metin

    for deneme in range(1, azami_deneme + 1):
        veri = llm.json_ayikla(ham)
        try:
            return sema.dogrula(veri), deneme
        except sema.SemaHatasi as hata:
            if deneme == azami_deneme:
                raise
            print(f'  [deneme {deneme}] sema bozuk: {hata}')
            print(f'  [deneme {deneme}] onarim turu calistiriliyor...')
            onarim = sema.onarim_istemi(ham, str(hata))
            ham = llm.sor(onarim, sicaklik=0.0,
                          senaryo='lab1_onarim').metin

for i in range(3):
    print(f'--- Cagri {i+1} ---')
    try:
        sonuc, deneme = semali_cikar(ISTEK_SEMALI)
        print(f'  BASARILI ({deneme}. denemede): {sonuc}')
    except sema.SemaHatasi as hata:
        print(f'  BASARISIZ: {hata}')
        print('  -> Bu kayit insana yonlendirilmeli.')
    print()

---
## 6. Guven esigi: mimari karar, model karari degil

Semada bir `guven` alani var. Modelin kendi guvenini bilmedigini
1. gunde konusmustuk, o yuzden bu deger tek basina yeterli degil.
Ama esik koymak yine de mimarinin elindeki en ucuz kaldiraclardan biri.

In [ ]:
ESIK = 0.60

ornekler = []
for i in range(5):
    c = llm.sor(ISTEK_SEMALI, sicaklik=0.7, senaryo='lab1_sema')
    veri = llm.json_ayikla(c.metin)
    try:
        ornekler.append(sema.dogrula(veri))
    except sema.SemaHatasi:
        pass

for kayit in ornekler:
    guven = kayit.get('guven', 0)
    yol = 'OTOMATIK AKIS' if guven >= ESIK else 'INSANA YONLENDIR'
    print(f'guven={guven:.2f}  ->  {yol}')

---
## Egzersiz

1. `ortak/sema.py` icindeki `TALEP_SEMASI`'na `musteri_id` alani ekleyin
   (bicim: `M-` ve dort rakam). Cikarimi tekrar calistirin, ne oluyor?
2. `ESIK` degerini 0.80 yapin. Kac kayit insana gidiyor?
3. `azami_deneme` degerini 1 yapin. Onarim turu olmadan basari orani
   ne kadar dusuyor?

---
## Alinacak ders

> Ciktiyi ayristirilabilir ve dogrulanabilir kilmadan sisteme sokamazsiniz.
> Sema bir bicimlendirme tercihi degil, sistemin girisindeki kilittir.